# Does an LSTM beat naive on GOOGL daily returns?

**An honest test.** This notebook is deliberately built so the LSTM *cannot* fool you.
The classic stock-prediction trap is to predict tomorrow's **price** and score with
RMSE — which rewards a model for echoing today's price (prices are ~99% persistent),
producing a beautiful-looking fit with zero real skill. We avoid that trap:

1. **Target is the return, not the price.** The loss sits on the change, not the level.
2. **Baselines run alongside.** A naive 'no change' / persistence baseline and a linear
   model. The LSTM has to *beat* them to mean anything.
3. **Directional + IC metrics, not RMSE-on-price.** We measure whether it gets the
   *direction and rank* right — the hard part — not whether it tracks the level.
4. **No-leakage split.** Strict chronological train/test; scaling fit on train only.

**Prior:** returns at a daily horizon on a single ticker are almost pure noise, and an
LSTM has thousands of parameters to overfit ~4,000 rows. The expected honest result is
**directional accuracy ~50%** (a coin flip) and **IC ~0** — i.e. no signal. Seeing that
yourself, with the fancy model, is the point.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LinearRegression
from scipy.stats import spearmanr

np.random.seed(42)
TICKER = "GOOGL"
LOOKBACK = 20      # days of history the LSTM sees per prediction
TEST_FRAC = 0.2    # last 20% is out-of-sample

## 1. Data — GOOGL daily, target = next-day LOG RETURN

We predict the **next day's log return**, computed from adjusted close (so splits and
dividends don't create fake jumps). This is the honest target: the model is scored on
the change, and it has no way to hide behind price persistence.

In [ ]:
import yfinance as yf

raw = yf.download(TICKER, start="2010-01-01", auto_adjust=True, progress=False)

# Recent yfinance returns MultiIndex columns like ('Close','GOOGL'). Flatten to
# single-level lowercase names so 'close', 'high', etc. work regardless of version.
if isinstance(raw.columns, pd.MultiIndex):
    raw.columns = raw.columns.get_level_values(0)
raw.columns = [str(c).lower() for c in raw.columns]

df = raw[["close", "high", "low", "open", "volume"]].dropna().copy()

# Target: NEXT day's log return. Features: today's return + simple technicals.
df["log_return"] = np.log(df["close"] / df["close"].shift(1))
df["target"] = df["log_return"].shift(-1)          # what we predict

# A few honest features (today's info only -- no look-ahead)
df["ret_1"] = df["log_return"]
df["ret_5"] = df["log_return"].rolling(5).mean()
df["vol_10"] = df["log_return"].rolling(10).std()
df["rng"] = (df["high"] - df["low"]) / df["close"]
df["vol_chg"] = df["volume"].pct_change().clip(-3, 3)

FEATURES = ["ret_1", "ret_5", "vol_10", "rng", "vol_chg"]
df = df.dropna()
print(f"{TICKER}: {len(df)} rows, {df.index.min().date()} -> {df.index.max().date()}")
df[["log_return", "target"] + FEATURES].tail()

## 2. No-leakage chronological split + scaling on train only

In [ ]:
split = int(len(df) * (1 - TEST_FRAC))
train_df, test_df = df.iloc[:split], df.iloc[split:]
print(f"train {len(train_df)}  |  test {len(test_df)}")
print(f"test period: {test_df.index.min().date()} -> {test_df.index.max().date()}")

# Scale features on TRAIN ONLY (fitting on all data would leak test stats).
scaler = StandardScaler().fit(train_df[FEATURES])
Xtr_flat = scaler.transform(train_df[FEATURES])
Xte_flat = scaler.transform(test_df[FEATURES])
ytr = train_df["target"].values
yte = test_df["target"].values

## 3. The honest scorecard

Every model is scored the same way, on the **out-of-sample** test set:

- **Directional accuracy** — % of days it gets up/down right. **50% = no skill.** This is
  the number that matters. Anything not clearly above 50% is noise.
- **IC (rank correlation)** between predicted and actual returns. **~0 = no signal.**
- **RMSE on the return** — shown only for completeness; it is *not* the headline, because
  a model that always predicts ~0 gets a great return-RMSE while having no skill.

In [ ]:
def scorecard(name, y_true, y_pred):
    y_true, y_pred = np.asarray(y_true), np.asarray(y_pred)
    dir_acc = np.mean(np.sign(y_true) == np.sign(y_pred))
    ic = spearmanr(y_true, y_pred).correlation if np.std(y_pred) > 0 else 0.0
    rmse = np.sqrt(np.mean((y_true - y_pred) ** 2))
    return {"model": name, "dir_acc": dir_acc, "IC": ic, "rmse": rmse}

results = []

## 4. Baselines — the bar the LSTM must clear

**Naive / persistence:** predict tomorrow's return = today's return. **Zero:** always
predict no change (the true 'no information' baseline). **Linear:** a plain regression
on the same features — if a linear model can't find signal, an LSTM finding 'signal' is
a red flag for overfitting.

In [ ]:
# Naive: tomorrow's return = today's return
naive_pred = test_df["ret_1"].values
results.append(scorecard("naive (persistence)", yte, naive_pred))

# Zero: always predict no change (pure no-information)
results.append(scorecard("zero (no change)", yte, np.zeros_like(yte)))

# Linear regression on the features
lin = LinearRegression().fit(Xtr_flat, ytr)
results.append(scorecard("linear", yte, lin.predict(Xte_flat)))

pd.DataFrame(results).round(4)

## 5. The LSTM

A standard LSTM over `LOOKBACK`-day windows of the features. Dropout to fight overfitting,
early stopping on a validation slice. This is a fair, competent LSTM — not a strawman.

*Requires `tensorflow`. If not installed: `pip install tensorflow`. The notebook falls
back to a note if it's missing.*

In [ ]:
def make_sequences(X_flat, y, lookback):
    """Turn flat (n, features) into (n-lookback, lookback, features) sequences."""
    Xs, ys = [], []
    for i in range(lookback, len(X_flat)):
        Xs.append(X_flat[i - lookback:i])
        ys.append(y[i])
    return np.array(Xs), np.array(ys)

try:
    import tensorflow as tf
    from tensorflow.keras.models import Sequential
    from tensorflow.keras.layers import LSTM, Dense, Dropout
    from tensorflow.keras.callbacks import EarlyStopping
    tf.random.set_seed(42)

    Xtr_seq, ytr_seq = make_sequences(Xtr_flat, ytr, LOOKBACK)
    Xte_seq, yte_seq = make_sequences(Xte_flat, yte, LOOKBACK)

    model = Sequential([
        LSTM(32, input_shape=(LOOKBACK, len(FEATURES)), return_sequences=True),
        Dropout(0.3),
        LSTM(16),
        Dropout(0.3),
        Dense(1),
    ])
    model.compile(optimizer="adam", loss="mse")
    es = EarlyStopping(patience=10, restore_best_weights=True)
    hist = model.fit(Xtr_seq, ytr_seq, validation_split=0.15, epochs=100,
                     batch_size=32, callbacks=[es], verbose=0)
    print(f"trained {len(hist.history['loss'])} epochs")

    lstm_pred = model.predict(Xte_seq, verbose=0).flatten()
    results.append(scorecard("LSTM", yte_seq, lstm_pred))
    lstm_ran = True
except ImportError:
    print("tensorflow not installed — run `pip install tensorflow` and re-run this cell.")
    lstm_ran = False

## 6. The verdict

In [ ]:
table = pd.DataFrame(results).round(4)
print(table.to_string(index=False))
print()

best_dir = table.loc[table['dir_acc'].idxmax()]
print(f"Best directional accuracy: {best_dir['model']} at {best_dir['dir_acc']:.1%}")
print()
if lstm_ran:
    lstm_row = table[table['model'] == 'LSTM'].iloc[0]
    naive_row = table[table['model'] == 'naive (persistence)'].iloc[0]
    print(f"LSTM directional accuracy : {lstm_row['dir_acc']:.1%}")
    print(f"LSTM IC                   : {lstm_row['IC']:+.4f}")
    print()
    if lstm_row['dir_acc'] <= 0.53 and abs(lstm_row['IC']) < 0.05:
        print("VERDICT: No signal. The LSTM is at ~coin-flip directional accuracy and ~0 IC.")
        print("It does NOT beat naive in any way that matters. This confirms — with the")
        print("fancy model, on real GOOGL data — that daily single-ticker returns are not")
        print("predictable here. Exactly the finding the project already built around.")
    elif lstm_row['dir_acc'] > 0.55:
        print("The LSTM shows above-chance directional accuracy. BEFORE believing it:")
        print("  - Re-run with a different seed; if it moves a lot, it's noise.")
        print("  - Check it survives transaction costs and holds on non-overlapping periods.")
        print("  - A single-ticker LSTM beating naive is more often overfitting than signal.")
    else:
        print("VERDICT: Marginal / ambiguous — consistent with noise. Not a basis to ship")
        print("a return signal.")

In [ ]:
# The persistence-trap demonstration: what it would have looked like on PRICE.
# This is the seductive-but-fake version, shown so you can see the difference.
if lstm_ran:
    fig, ax = plt.subplots(1, 2, figsize=(14, 4))

    # LEFT: predicted vs actual RETURNS (the honest view — looks like noise, because it is)
    ax[0].scatter(yte_seq, lstm_pred, s=8, alpha=0.4)
    ax[0].axhline(0, color='grey', lw=0.5); ax[0].axvline(0, color='grey', lw=0.5)
    ax[0].set_xlabel('actual return'); ax[0].set_ylabel('predicted return')
    ax[0].set_title('Honest view: predicted vs actual RETURNS\n(a blob = no signal)')

    # RIGHT: reconstruct PRICE from predictions (the trap — looks great, means nothing)
    test_prices = test_df['close'].values[LOOKBACK:]
    pred_price = test_prices[:-1] * np.exp(lstm_pred[1:])  # naive price reconstruction
    ax[1].plot(test_prices[1:], label='actual price', lw=1)
    ax[1].plot(pred_price, label='"predicted" price', lw=1, alpha=0.7)
    ax[1].legend(); ax[1].set_title('The TRAP: price view\n(looks accurate — but it is just echoing yesterday)')
    plt.tight_layout(); plt.show()

    print("Left looks like noise (honest). Right looks accurate (seductive, fake).")
    print("Same model. The price view hides the lack of skill behind persistence —")
    print("which is exactly the trap this notebook is built to expose.")